In [1]:
import pandas as pd

In [2]:
path = "C:/Users/vihaa/Desktop/Solar Panel/Solar Panel Fault Detection"

path = r"C:\Users\vihaa\Desktop\Solar Panel\Solar Panel Fault Detection
unicodeescape

In [3]:
solar = pd.read_csv(path + "/backend/data/timeseries/raw/Solar_Energy_Generation.csv")
weather=pd.read_csv(path+"/backend/data/timeseries/raw/Weather_Data_reordered_all.csv")

In [4]:
print(solar.head())
solar.shape
#Basic details about the df

   CampusKey  SiteKey            Timestamp  SolarGeneration
0          2        1  2020-01-01 00:15:00              NaN
1          2        1  2020-01-01 00:30:00              NaN
2          2        1  2020-01-01 00:45:00              NaN
3          2        1  2020-01-01 01:00:00              NaN
4          2        1  2020-01-01 01:15:00              NaN


(2731946, 4)

Visualise Dataset (Used Kaggle Summary)

In [5]:
solar['Timestamp']=pd.to_datetime(solar['Timestamp'])
#Can use timestamp properties existing in pandas

In [7]:
site1=solar[solar["SiteKey"]==1].copy()

In [10]:
site1.isna().sum()

CampusKey              0
SiteKey                0
Timestamp              0
SolarGeneration    41746
dtype: int64

In [11]:
(site1['SolarGeneration'].isna().sum() / len(site1)) * 100

np.float64(52.6305172783318)

In [12]:
counts = site1.groupby(site1['Timestamp'].dt.hour)['SolarGeneration'].agg(
    total='size',
    nan_count=lambda x: x.isna().sum(),
    not_nan_count=lambda x: x.notna().sum()
)

In [13]:
print(counts)

           total  nan_count  not_nan_count
Timestamp                                 
0           3308       3308              0
1           3320       3320              0
2           3321       3321              0
3           3323       3323              0
4           3321       3321              0
5           3316       3316              0
6           3316       2730            586
7           3315        987           2328
8           3310        115           3195
9           3308         84           3224
10          3308         68           3240
11          3308        143           3165
12          3305        315           2990
13          3301        362           2939
14          3300        222           3078
15          3300         73           3227
16          3302         53           3249
17          3301        657           2644
18          3296       1471           1825
19          3296       1813           1483
20          3296       2896            400
21         

Important for analysing data
Month	Sunrise	Sunset	Day Length
Jan	earliest	latest	longest
Feb	slightly later	slightly earlier	↓
Mar	increasing	decreasing	medium
Apr	later	earlier	↓
May	later	earlier	↓
Jun	latest	earliest	shortest
Jul	slightly earlier	slightly later	↑
Aug	earlier	later	↑
Sep	balanced	balanced	medium
Oct	earlier	later	↑
Nov	early	late	↑
Dec	earliest	latest	longest

In [ ]:
time_diff = site1['Timestamp'].sort_values().diff()
#Sorting inside that expression does NOT affect site1; it works on a temporary copy.
print(time_diff.value_counts())
'''Helps detect data frequency (e.g., every 5 min, 10 min)
Finds missing data / irregular gaps
Helps in resampling & ML preprocessing'''

In [14]:
site1 = site1.set_index('Timestamp')
site1 = site1.asfreq('15T')
#Converts Timestamp column into the index and sorts it 
# and Convert to time-index → enforce 15-minute intervals → fill missing timestamps with NaN

C:\Users\vihaa\AppData\Local\Temp\ipykernel_78636\4014107380.py:2: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  site1 = site1.asfreq('15T')


(Option that we are not pursuing right now) Remove night ✅. We are not doing this right now. Will that help model?

In [16]:
import numpy as np

def process_day(df):
    df = df.copy()
    
    values = df['SolarGeneration'].values
    valid_idx = np.where(values > 0)[0]
    
    if len(valid_idx) == 0:
        df['SolarGeneration'] = 0
        return df
    #Entire day has 0 solar generation 
    first_valid = valid_idx[0]
    last_valid = valid_idx[-1]
    
    col_idx = df.columns.get_loc('SolarGeneration')
    
    df.iloc[:first_valid, col_idx] = 0
    df.iloc[last_valid+1:, col_idx] = 0
    
    middle = df.iloc[first_valid:last_valid+1].copy()
    
    middle['SolarGeneration'] = middle['SolarGeneration'].replace(0, np.nan)
    middle['SolarGeneration'] = middle['SolarGeneration'].interpolate(
        method='linear',
        limit_direction='both'
    )
    
    df.iloc[first_valid:last_valid+1] = middle
    #Only middle values are interpolated
    return df

# ✅ Just ensure sorted
site1 = site1.sort_index()

# ✅ Apply preprocessing
site1 = site1.groupby(site1.index.date, group_keys=False).apply(process_day)

# ✅ Final cleanup
site1['SolarGeneration'] = site1['SolarGeneration'].fillna(0)
site1[['CampusKey', 'SiteKey']] = site1[['CampusKey', 'SiteKey']].ffill().bfill()

In [17]:
site1.isna().sum()

CampusKey          0
SiteKey            0
SolarGeneration    0
dtype: int64

In [18]:
site1

,CampusKey,SiteKey,SolarGeneration
Timestamp,,,
2020-01-01 00:15:00,2.0,1.0,0.0
2020-01-01 00:30:00,2.0,1.0,0.0
2020-01-01 00:45:00,2.0,1.0,0.0
2020-01-01 01:00:00,2.0,1.0,0.0
2020-01-01 01:15:00,2.0,1.0,0.0
...,...,...,...
2022-04-23 22:45:00,2.0,1.0,0.0
2022-04-23 23:00:00,2.0,1.0,0.0
2022-04-23 23:15:00,2.0,1.0,0.0


In [19]:
site1['hour'] = site1.index.hour
site1['day'] = site1.index.day
site1['month'] = site1.index.month
site1['day_of_week'] = site1.index.dayofweek
#We extract hour/day/month etc. so the ML model can learn time-based patterns in solar generation

In [20]:
site1

,CampusKey,SiteKey,SolarGeneration,hour,day,month,day_of_week
Timestamp,,,,,,,
2020-01-01 00:15:00,2.0,1.0,0.0,0,1,1,2
2020-01-01 00:30:00,2.0,1.0,0.0,0,1,1,2
2020-01-01 00:45:00,2.0,1.0,0.0,0,1,1,2
2020-01-01 01:00:00,2.0,1.0,0.0,1,1,1,2
2020-01-01 01:15:00,2.0,1.0,0.0,1,1,1,2
...,...,...,...,...,...,...,...
2022-04-23 22:45:00,2.0,1.0,0.0,22,23,4,5
2022-04-23 23:00:00,2.0,1.0,0.0,23,23,4,5
2022-04-23 23:15:00,2.0,1.0,0.0,23,23,4,5


Different Dataset

In [21]:
weather['Timestamp'] = pd.to_datetime(weather['Timestamp'])

In [22]:
weather.isna().sum()

CampusKey                   0
Timestamp                   0
ApparentTemperature    107113
AirTemperature         107113
DewPointTemperature    107113
RelativeHumidity       107113
WindSpeed              162890
WindDirection          162890
dtype: int64

In [23]:
weather.head()

,CampusKey,Timestamp,ApparentTemperature,AirTemperature,DewPointTemperature,RelativeHumidity,WindSpeed,WindDirection
0,1,2020-01-01 00:00:00,13.666667,13.880000,8.960000,72.400000,0.000000,188.133333
1,1,2020-01-01 00:15:00,13.206667,13.666667,9.040000,73.466667,1.200000,203.866667
2,1,2020-01-01 00:30:00,12.840000,13.553333,9.053333,74.000000,2.520000,222.800000
3,1,2020-01-01 00:45:00,12.113333,13.506667,9.100000,74.466667,5.986667,231.133333
4,1,2020-01-01 01:00:00,11.946667,13.260000,9.266667,76.533333,5.946667,247.866667


In [27]:
wcampus2=weather[weather["CampusKey"]==2].copy()

In [28]:
wcampus2.head()

,CampusKey,Timestamp,ApparentTemperature,AirTemperature,DewPointTemperature,RelativeHumidity,WindSpeed,WindDirection
81017,2,2020-01-01 00:00:00,17.986667,20.140000,9.173333,49.200000,10.120000,180.666667
81018,2,2020-01-01 00:15:00,18.560000,20.300000,9.600000,50.266667,8.680000,152.666667
81019,2,2020-01-01 00:30:00,18.146667,19.840000,9.340000,50.666667,7.933333,157.533333
81020,2,2020-01-01 00:45:00,18.140000,19.886667,9.486667,51.000000,8.440000,147.600000
81021,2,2020-01-01 01:00:00,17.693333,19.540000,9.180000,51.133333,8.680000,151.333333


In [29]:
wcampus2.isna().sum()

CampusKey                  0
Timestamp                  0
ApparentTemperature    12763
AirTemperature         12763
DewPointTemperature    12763
RelativeHumidity       12763
WindSpeed              26042
WindDirection          26042
dtype: int64

In [30]:
wcampus2['Timestamp'].is_monotonic_increasing

True

In [31]:
wcampus2 = wcampus2.sort_values('Timestamp')

In [32]:
w_time_diff = wcampus2['Timestamp'].sort_values().diff()

print(w_time_diff.value_counts())

Timestamp
0 days 00:15:00    81016
Name: count, dtype: int64


In [33]:
wcampus2 = wcampus2.set_index('Timestamp')

In [38]:
wcampus2.index.duplicated().any()

np.False_

In [34]:
#duplicate
wcampus2 = wcampus2[~wcampus2.index.duplicated(keep='first')]
#This line removes duplicate timestamps (index values) and keeps only the first occurrence.

In [35]:
wcampus2.index.duplicated().sum()
#Check this before

np.int64(0)

In [39]:
full_time = pd.date_range(
    start=wcampus2.index.min(),
    end=wcampus2.index.max(),
    freq='15T'
)

wcampus2 = wcampus2.reindex(full_time)
#I think here we have the entire time

C:\Users\vihaa\AppData\Local\Temp\ipykernel_78636\4143319997.py:1: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  full_time = pd.date_range(


In [40]:
wcampus2.isna().sum()

CampusKey                  0
ApparentTemperature    12763
AirTemperature         12763
DewPointTemperature    12763
RelativeHumidity       12763
WindSpeed              26042
WindDirection          26042
dtype: int64

In [41]:
wcampus2 = wcampus2.interpolate(method='time')

In [42]:
wcampus2.isna().sum()

CampusKey              0
ApparentTemperature    0
AirTemperature         0
DewPointTemperature    0
RelativeHumidity       0
WindSpeed              0
WindDirection          0
dtype: int64

In [ ]:
#data = pd.merge(site1, wcampus2, on='Timestamp')

In [ ]:
features = [
    'AirTemperature',
    'RelativeHumidity',
    'WindSpeed',
    'lag_1',
    'lag_4',
    'rolling_mean_4',
    'hour',
    'month'
]

X = data[features]
y = data['SolarGeneration']

In [ ]:
X.isna().sum()

In [ ]:
split_index = int(len(data) * 0.8)

In [ ]:
train = data.iloc[:split_index]
test = data.iloc[split_index:]

In [ ]:
X_train = train[features]
y_train = train['SolarGeneration']

X_test = test[features]
y_test = test['SolarGeneration']

In [ ]:
print(train['Timestamp'].min(), train['Timestamp'].max())
print(test['Timestamp'].min(), test['Timestamp'].max())

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)

model.fit(X_train, y_train)

In [ ]:
print("Model trained successfully")

In [ ]:
pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print("MAE:", mae)
print("RMSE:", rmse)

In [ ]:
print(y_test.describe())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(y_test.values[:200], label='Actual')
plt.plot(pred[:200], label='Predicted')
plt.legend()
plt.title("Actual vs Predicted")
plt.show()